# Dual Next-Event Prediction Notebook

This notebook trains and compares dual-head models that predict:
- next activity (`concept:name`)
- next lifecycle transition (`lifecycle:transition`)

It runs two lifecycle data modes:
1. `start_complete` (only start/complete events)
2. `full_lifecycle` (all lifecycle transitions)

For each mode, it compares:
- `baseline`
- `balanced` (class-balanced sample weighting)

The final ranking is based on a balanced score.

In [13]:
from pathlib import Path
import sys
import json
import pandas as pd

# Make notebook imports robust to current working directory
cwd = Path.cwd().resolve()
if (cwd / "next_activity_prediction_lifecycle_dual").exists():
    repo_root = cwd
elif (cwd.parent / "next_activity_prediction_lifecycle_dual").exists():
    repo_root = cwd.parent
else:
    repo_root = cwd

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from next_activity_prediction_lifecycle_dual.config import DualPredictionConfig
from next_activity_prediction_lifecycle_dual.trainer import run_full_experiment

print("Notebook cwd:", cwd)
print("Repo root used for imports:", repo_root)

Notebook cwd: D:\Repos\process-simulation-engine-1\next_activity_prediction_lifecycle_dual
Repo root used for imports: D:\Repos\process-simulation-engine-1\next_activity_prediction_lifecycle_dual


In [14]:
# Set your event log path here
# Examples:
# EVENT_LOG_PATH = "Dataset/BPIC12.xes"
# EVENT_LOG_PATH = "Dataset/your_log.csv"
# Set your event log path here
EVENT_LOG_PATH = fr"..\Dataset\BPI Challenge 2017.xes"

MODEL_ROOT = "next_activity_prediction_lifecycle_dual/models"

# Resolve relative path from detected repo root (set in previous cell)
log_path_obj = Path(EVENT_LOG_PATH)
if not log_path_obj.is_absolute():
    log_path_obj = repo_root / log_path_obj

if not log_path_obj.exists():
    raise FileNotFoundError(f"Event log not found: {log_path_obj}")

EVENT_LOG_PATH = str(log_path_obj)

config = DualPredictionConfig(
    sequence_length=50,
    embedding_dim=96,
    lstm_units=192,
    lstm_layers=2,
    dropout_rate=0.25,
    batch_size=64,
    learning_rate=0.001,
    epochs=1,
    validation_split=0.1,
    early_stopping_patience=8,
    model_root=MODEL_ROOT,
)

print("Using log:", EVENT_LOG_PATH)
print("Output root:", config.model_root)

Using log: D:\Repos\process-simulation-engine-1\next_activity_prediction_lifecycle_dual\..\Dataset\BPI Challenge 2017.xes
Output root: next_activity_prediction_lifecycle_dual\models


In [15]:
# Train all 4 combinations:
# - start_complete x baseline
# - start_complete x balanced
# - full_lifecycle x baseline
# - full_lifecycle x balanced
summary = run_full_experiment(log_path=EVENT_LOG_PATH, config=config)
summary["best_model"]

8488/8488 ━━━━━━━━━━━━━━━━━━━━ 0s 178ms/step - activity_output_loss: 0.6557 - activity_output_sparse_categorical_accuracy: 0.7699 - lifecycle_output_loss: 0.3028 - lifecycle_output_sparse_categorical_accuracy: 0.8562 - loss: 0.9585
Epoch 1: val_loss improved from None to 0.61047, saving model to next_activity_prediction_lifecycle_dual\models\start_complete\baseline\checkpoints\best_model.keras

Epoch 1: finished saving model to next_activity_prediction_lifecycle_dual\models\start_complete\baseline\checkpoints\best_model.keras
8488/8488 ━━━━━━━━━━━━━━━━━━━━ 1555s 183ms/step - activity_output_loss: 0.4680 - activity_output_sparse_categorical_accuracy: 0.8246 - lifecycle_output_loss: 0.2495 - lifecycle_output_sparse_categorical_accuracy: 0.8745 - loss: 0.7175 - val_activity_output_loss: 0.3881 - val_activity_output_sparse_categorical_accuracy: 0.8444 - val_lifecycle_output_loss: 0.2223 - val_lifecycle_output_sparse_categorical_accuracy: 0.8857 - val_loss: 0.6105 - learning_rate: 0.0010
Re

{'mode': 'full_lifecycle',
 'methodology': 'baseline',
 'metrics': {'activity_accuracy': 0.9188035965298976,
  'activity_macro_f1': 0.7731343798372036,
  'lifecycle_accuracy': 0.886273466026766,
  'lifecycle_macro_f1': 0.8862035019098388,
  'joint_accuracy': 0.8693804220349838,
  'balanced_score': 0.8429061012606754},
 'model_dir': 'next_activity_prediction_lifecycle_dual\\models\\full_lifecycle\\baseline'}

In [16]:
summary_path = Path(MODEL_ROOT) / "comparison_summary.json"
with open(summary_path, "r", encoding="utf-8") as f:
    summary_from_file = json.load(f)

rows = []
for item in summary_from_file["all_results"]:
    m = item["metrics"]
    rows.append({
        "mode": item["mode"],
        "methodology": item["methodology"],
        "balanced_score": m["balanced_score"],
        "joint_accuracy": m["joint_accuracy"],
        "activity_macro_f1": m["activity_macro_f1"],
        "lifecycle_macro_f1": m["lifecycle_macro_f1"],
        "activity_accuracy": m["activity_accuracy"],
        "lifecycle_accuracy": m["lifecycle_accuracy"],
        "model_dir": item["model_dir"],
    })

results_df = pd.DataFrame(rows).sort_values("balanced_score", ascending=False)
results_df

,mode,methodology,balanced_score,joint_accuracy,activity_macro_f1,lifecycle_macro_f1,activity_accuracy,lifecycle_accuracy,model_dir
0,full_lifecycle,baseline,0.842906,0.869380,0.773134,0.886204,0.918804,0.886273,next_activity_prediction_lifecycle_dual\models...
1,full_lifecycle,balanced,0.835449,0.850716,0.776320,0.879312,0.910236,0.874978,next_activity_prediction_lifecycle_dual\models...
2,start_complete,baseline,0.799465,0.832704,0.698430,0.867262,0.844385,0.885741,next_activity_prediction_lifecycle_dual\models...
3,start_complete,balanced,0.795177,0.825132,0.696980,0.863419,0.841585,0.877738,next_activity_prediction_lifecycle_dual\models...


In [17]:
# Simple view: best model per lifecycle mode
best_per_mode = (
    results_df.sort_values("balanced_score", ascending=False)
    .groupby("mode", as_index=False)
    .first()
)
best_per_mode

,mode,methodology,balanced_score,joint_accuracy,activity_macro_f1,lifecycle_macro_f1,activity_accuracy,lifecycle_accuracy,model_dir
0,full_lifecycle,baseline,0.842906,0.869380,0.773134,0.886204,0.918804,0.886273,next_activity_prediction_lifecycle_dual\models...
1,start_complete,baseline,0.799465,0.832704,0.698430,0.867262,0.844385,0.885741,next_activity_prediction_lifecycle_dual\models...


## Notes

- Use `balanced_score` to choose the most well-balanced model.
- If you need faster iterations, reduce `epochs` in the config cell.
- You can rerun only the training cell after changing hyperparameters.
- Saved artifacts are in `next_activity_prediction_lifecycle_dual/models`.